# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [1]:
import pandas as pd

---
## 2. Load data

In [2]:
df = pd.read_csv('./clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [3]:
original_columns = df.columns.tolist()

In [4]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [5]:
price_df = pd.read_csv('price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [6]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

In [7]:
# ============================================
# Additional Feature Engineering
# ============================================

# Contract duration
df["contract_duration_days"] = (
    df["date_end"] - df["date_activ"]
).dt.days

df["contract_duration_years"] = (
    df["contract_duration_days"] / 365
)

# Time since product modification
df["days_since_modif_prod"] = (
    df["date_end"] - df["date_modif_prod"]
).dt.days

# Time between product modification and renewal
df["days_modif_to_renewal"] = (
    df["date_renewal"] - df["date_modif_prod"]
).dt.days

# Time between activation and renewal
df["days_activ_to_renewal"] = (
    df["date_renewal"] - df["date_activ"]
).dt.days

# Total consumption
df["total_consumption_12m"] = (
    df["cons_12m"] + df["cons_gas_12m"]
)

# Electricity-to-gas consumption ratio
df["electricity_gas_ratio"] = (
    df["cons_12m"] / (df["cons_gas_12m"] + 1)
)

# Difference between annual and recent consumption
df["consumption_change_12m_vs_last_month"] = (
    df["cons_12m"] - df["cons_last_month"]
)

# Average monthly consumption
df["avg_monthly_consumption"] = (
    df["total_consumption_12m"] / 12
)

# ============================================
# Check results
# ============================================

new_features = [
    col for col in df.columns
    if col not in original_columns
]

print("Original number of columns:", len(original_columns))
print("New number of columns:", df.shape[1])

print("\nNew features:")
print(new_features)

print("\nPreview:")
display(df[new_features].head())

Original number of columns: 44
New number of columns: 53

New features:
['contract_duration_days', 'contract_duration_years', 'days_since_modif_prod', 'days_modif_to_renewal', 'days_activ_to_renewal', 'total_consumption_12m', 'electricity_gas_ratio', 'consumption_change_12m_vs_last_month', 'avg_monthly_consumption']

Preview:


,contract_duration_days,contract_duration_years,days_since_modif_prod,days_modif_to_renewal,days_activ_to_renewal,total_consumption_12m,electricity_gas_ratio,consumption_change_12m_vs_last_month,avg_monthly_consumption
0,1096,3.002740,227,-131,738,54946,0.0,0,4578.833333
1,2566,7.030137,2566,2201,2201,4660,4660.0,4660,388.333333
2,2192,6.005479,2192,1827,1827,544,544.0,544,45.333333
3,2192,6.005479,2192,1827,1827,1584,1584.0,1584,132.000000
4,2245,6.150685,2245,1881,1881,4425,4425.0,3899,368.750000


In [8]:
print("Original number of columns: 44")
print("New number of columns:", df.shape[1])
print("\nNew features:")
print(new_features)

Original number of columns: 44
New number of columns: 53

New features:
['contract_duration_days', 'contract_duration_years', 'days_since_modif_prod', 'days_modif_to_renewal', 'days_activ_to_renewal', 'total_consumption_12m', 'electricity_gas_ratio', 'consumption_change_12m_vs_last_month', 'avg_monthly_consumption']


In [9]:
import os

print(os.listdir("."))

['-fK8Xt4_QGyHNWNBiTrTzw_fbd420150db6471ba448fb29afa673f1_Activity-Template_-RACI-Matrix.docx', '.ipynb_checkpoints', '26-07-2026.pdf', '3151910-operations-research-theory-and-applications-by-j.-k.-sharma-z-lib.org_ (1).pdf', '3151910-operations-research-theory-and-applications-by-j.-k.-sharma-z-lib.org_.pdf', '475789025_2348790942165218_7409930153597651614_n (1).jpg', '475789025_2348790942165218_7409930153597651614_n.jpg', '4th Sem Result Srabana Mukherjee.pdf', '5.14 Project 1.pdf', 'Adhaar_Card.pdf', 'Advertisement.pdf', "Analysis of Sir Giri's Paper.docx", 'Analysis_and_Applications_of_2D_Discrete_Fourier_Transform_in_Image_Denoising_and_Edge_Detection (1).pdf', 'Analysis_and_Applications_of_2D_Discrete_Fourier_Transform_in_Image_Denoising_and_Edge_Detection.pdf', 'Anamika Dash Thesis-19-35.pdf', 'Anamika Prsentation on 03_02_2026.pdf', 'announcements_1771577018780_832947a8-5a35-4147-9fea-8e181ce13fd0_Phase_I_of_National_Internship_in_Official_Statistics_(NIOS)_2026-27_of_MoSPI.pdf

In [10]:
# ============================================
# Additional Feature Engineering
# ============================================

import pandas as pd
import numpy as np

# ------------------------------------------------
# 1. LOAD HISTORICAL PRICE DATA
# ------------------------------------------------

# Change this filename ONLY if your price CSV has a different name.
price_df = pd.read_csv("./price_data.csv")

price_df["price_date"] = pd.to_datetime(
    price_df["price_date"],
    format="%Y-%m-%d"
)

# ------------------------------------------------
# 2. CREATE PRICE DIFFERENCE FEATURES
# ------------------------------------------------

# Difference between peak and off-peak power prices
price_df["power_price_peak_difference"] = (
    price_df["price_peak_var"] -
    price_df["price_off_peak_var"]
)

# Difference between mid-peak and off-peak power prices
price_df["power_price_mid_peak_difference"] = (
    price_df["price_mid_peak_var"] -
    price_df["price_off_peak_var"]
)

# Difference between peak and mid-peak power prices
price_df["power_price_peak_mid_difference"] = (
    price_df["price_peak_var"] -
    price_df["price_mid_peak_var"]
)

# Average price difference across the three periods
price_df["average_price_difference"] = (
    price_df[
        [
            "power_price_peak_difference",
            "power_price_mid_peak_difference",
            "power_price_peak_mid_difference"
        ]
    ]
    .abs()
    .mean(axis=1)
)

# Maximum price difference across the three periods
price_df["maximum_price_difference"] = (
    price_df[
        [
            "power_price_peak_difference",
            "power_price_mid_peak_difference",
            "power_price_peak_mid_difference"
        ]
    ]
    .abs()
    .max(axis=1)
)

# ------------------------------------------------
# 3. AGGREGATE PRICE FEATURES BY CUSTOMER
# ------------------------------------------------

price_features = (
    price_df
    .groupby("id")
    .agg(
        average_price_difference=("average_price_difference", "mean"),
        maximum_price_difference=("maximum_price_difference", "max")
    )
    .reset_index()
)

# ------------------------------------------------
# 4. MERGE PRICE FEATURES WITH CUSTOMER DATA
# ------------------------------------------------

df = df.merge(
    price_features,
    on="id",
    how="left"
)

# ------------------------------------------------
# 5. DATE FEATURE ENGINEERING
# ------------------------------------------------

date_columns = [
    "date_activ",
    "date_end",
    "date_modif_prod",
    "date_renewal"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Extract month from activation date
if "date_activ" in df.columns:
    df["activation_month"] = df["date_activ"].dt.month

# Extract month from renewal date
if "date_renewal" in df.columns:
    df["renewal_month"] = df["date_renewal"].dt.month

# Extract month from product modification date
if "date_modif_prod" in df.columns:
    df["modification_month"] = df["date_modif_prod"].dt.month

# Extract month from contract end date
if "date_end" in df.columns:
    df["end_month"] = df["date_end"].dt.month

# Remove raw date columns
df = df.drop(columns=date_columns, errors="ignore")

# ------------------------------------------------
# 6. CONVERT BOOLEAN COLUMNS TO BINARY
# ------------------------------------------------

boolean_columns = df.select_dtypes(include=["bool"]).columns

for col in boolean_columns:
    df[col] = df[col].astype(int)

# Also handle boolean-like object columns
for col in df.select_dtypes(include=["object"]).columns:
    values = set(df[col].dropna().unique())

    if values.issubset({"True", "False"}):
        df[col] = df[col].map({
            "True": 1,
            "False": 0
        })

# ------------------------------------------------
# 7. CONVERT CATEGORICAL VARIABLES TO DUMMY VARIABLES
# ------------------------------------------------

categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()

# Do not encode the customer ID as a categorical feature
if "id" in categorical_columns:
    categorical_columns.remove("id")

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

# ------------------------------------------------
# 8. CHECK THE RESULT
# ------------------------------------------------

print("Final number of columns:", df.shape[1])

print("\nRemaining non-numeric columns:")
print(df.select_dtypes(exclude=["number"]).columns.tolist())

print("\nFinal data shape:")
print(df.shape)

print("\nFirst 5 rows:")
display(df.head())

Final number of columns: 65

Remaining non-numeric columns:
['id']

Final data shape:
(14606, 65)

First 5 rows:


C:\Users\sraba\AppData\Local\Temp\ipykernel_6584\1192461836.py:136: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]).columns:
C:\Users\sraba\AppData\Local\Temp\ipykernel_6584\1192461836.py:149: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_gu

,id,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,...,channel_sales_foosdfpfkusacimwkcsosbicdxkicaua,channel_sales_lmkebamcaaclubfxadlmueccxoimlema,channel_sales_sddiedcslfslkckwlfkdpoeeailfpeds,channel_sales_usilxuppasemubllopkaafesmlibmsdf,has_gas_t,origin_up_ewxeelcelemmiwuafmddpobolfuxioce,origin_up_kamkkxfxxuwbdslkwifmmcsiusiuosws,origin_up_ldkssxwpmemidmecebumciepifcamkci,origin_up_lxidpiddsbxsbosboudacockeimpuepw,origin_up_usapbepcfoloekilkwsdiboslwaxobdp
0,24011ae4ebbe3035111d65fa7c15bc57,0,54946,0,0.00,0,0.0,1.78,0.114481,0.098142,...,1,0,0,0,1,0,0,0,1,0
1,d29c2c54acc38ff3c0614d0a653813dd,4660,0,0,189.95,0,0.0,16.27,0.145711,0.000000,...,0,0,0,0,0,0,1,0,0,0
2,764c75f661154dac3a6c254cd082ea7d,544,0,0,47.96,0,0.0,38.72,0.165794,0.087899,...,1,0,0,0,0,0,1,0,0,0
3,bba03439a292a1e166f80264c16191cb,1584,0,0,240.04,0,0.0,19.83,0.146694,0.000000,...,0,1,0,0,0,0,1,0,0,0
4,149d57cf92fc41cf94415803a877cb4b,4425,0,526,445.75,526,0.0,131.73,0.116900,0.100015,...,0,0,0,0,0,0,1,0,0,0


In [11]:
# Remove customer ID because it is only an identifier
df = df.drop(columns=["id"])

print("Final number of columns:", df.shape[1])

print("Remaining non-numeric columns:")
print(df.select_dtypes(exclude=["number"]).columns.tolist())

print("\nFinal data shape:")
print(df.shape)

print("\nFirst 5 rows:")
display(df.head())

Final number of columns: 64
Remaining non-numeric columns:
[]

Final data shape:
(14606, 64)

First 5 rows:


,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_pow_off_peak,...,channel_sales_foosdfpfkusacimwkcsosbicdxkicaua,channel_sales_lmkebamcaaclubfxadlmueccxoimlema,channel_sales_sddiedcslfslkckwlfkdpoeeailfpeds,channel_sales_usilxuppasemubllopkaafesmlibmsdf,has_gas_t,origin_up_ewxeelcelemmiwuafmddpobolfuxioce,origin_up_kamkkxfxxuwbdslkwifmmcsiusiuosws,origin_up_ldkssxwpmemidmecebumciepifcamkci,origin_up_lxidpiddsbxsbosboudacockeimpuepw,origin_up_usapbepcfoloekilkwsdiboslwaxobdp
0,0,54946,0,0.00,0,0.0,1.78,0.114481,0.098142,40.606701,...,1,0,0,0,1,0,0,0,1,0
1,4660,0,0,189.95,0,0.0,16.27,0.145711,0.000000,44.311378,...,0,0,0,0,0,0,1,0,0,0
2,544,0,0,47.96,0,0.0,38.72,0.165794,0.087899,44.311378,...,1,0,0,0,0,0,1,0,0,0
3,1584,0,0,240.04,0,0.0,19.83,0.146694,0.000000,44.311378,...,0,1,0,0,0,0,1,0,0,0
4,4425,0,526,445.75,526,0.0,131.73,0.116900,0.100015,40.606701,...,0,0,0,0,0,0,1,0,0,0
